## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

df_high = pd.read_csv('data_resources/high_popularity_spotify_data.csv')

## The Expanded Lexicon

In [2]:
lexicon_db = {
    "Melancholy": [ # หม่นหมอง, R&B ช้า, Pop เศร้า
        'เจ็บ', 'ร้องไห้', 'ลืม', 'ทิ้ง', 'เศร้า', 'น้ำตา', 'ทรมาน', 'อ้อนวอน', 'แตกสลาย', 'อ้างว้าง',
        'sad', 'cry', 'broken', 'hurt', 'alone', 'tears', 'pain', 'fade', 'lonely'
    ],
    "Upbeat_Party": [ # สนุก, Dancehall, Pop เร็ว
        'สนุก', 'รัก', 'ยิ้ม', 'ปาร์ตี้', 'เต้น', 'สดใส', 'ชนแก้ว', 'เมา', 'สุดเหวี่ยง', 'คืนนี้', 'ฉลอง',
        'happy', 'love', 'smile', 'party', 'dance', 'tonight', 'vibes', 'cheers', 'high'
    ],
    "Aggressive_Flex": [ # ดุดัน, อวดรวย, Trap, Hip-Hop
        'รวย', 'เดือด', 'ของจริง', 'รันวงการ', 'แชมป์', 'สู้', 'ศัตรู', 'ทอง', 'เงิน', 'อำนาจ',
        'flex', 'money', 'hater', 'gang', 'boss', 'rich', 'grind', 'hustle', 'hood', 'crew', 'opps'
    ],
    "Seductive_Romance": [ # เซ็กซี่, ดึงดูด, R&B สมัยใหม่
        'เสน่ห์', 'หลง', 'จูบ', 'สัมผัส', 'ร้อนแรง', 'คืนนี้', 'ต้องการ', 'กลิ่นหอม', 'สบตา',
        'sexy', 'shawty', 'my lady', 'baby', 'kiss', 'touch', 'desire', 'body', 'babe'
    ]
}

## Thai Tokenization & TF-IDF Logic Engine

In [3]:
def clean_and_tokenize(text):
    text = text.lower()
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
    return tokens

def analyze_vibe_score(lyrics_text):
    """
    แนวคิด:
    1. รับเนื้อเพลงมาตัดคำ
    2. เช็คว่ามีคำไปตกอยู่ในหมวดไหนมากที่สุด
    3. (Optional) คุณสามารถแทรกโค้ด TfidfVectorizer ตรงนี้ เพื่อให้น้ำหนักคำที่หายากมากกว่าคำทั่วไปได้
    """
    tokens = clean_and_tokenize(lyrics_text)

    scores = {category: 0 for category in lexicon_db.keys()}

    for word in tokens:
        for category, keywords in lexicon_db.items():
            if word in keywords:
                scores[category] += 1

    best_vibe = max(scores, key=scores.get)
    return best_vibe, scores

## Song Arrangement & Dynamic Analysis

In [4]:
def evaluate_song_structure(song_sections):
    """
    รับ Input เป็น List ของ Dictionary เช่น:
    [{"type": "Intro", "lyrics": "..."}, {"type": "Hook", "lyrics": "..."}]
    """
    insights = []
    section_types = [sec['type'] for sec in song_sections]

    if "Hook" not in section_types:
        insights.append("🚨 Warning: เพลงนี้ไม่มีท่อน Hook! ซึ่งเป็นท่อนสำคัญที่สุดในการทำให้เพลงติดหู (Earworm)")

    if len(section_types) > 6:
        insights.append("💡 Note: โครงสร้างเพลงค่อนข้างยาว อาจจะลองตัดท่อนที่เยิ่นเย้อออกเพื่อให้เพลงกระชับขึ้น")

    return insights

## The Recommendation Engine

In [5]:
def generate_producer_brief(genre, subgenre, detected_vibe, df_high):
    """
    ใช้ Logic กรอง DataFrame และหาสถิติ (Median, Mode)
    """

    target_music_mode = 0 if detected_vibe in ["Melancholy", "Aggressive_Flex"] else 1

    rec_tempo = 110 # trend_data['tempo'].median()
    rec_key = 1     # trend_data['key'].mode()[0]

    return {
        "Target_Tempo": rec_tempo,
        "Target_Key": rec_key,
        "Target_Mode": "Minor" if target_music_mode == 0 else "Major"
    }

## Master Pipeline Testing

In [6]:
mock_song_input = [
    {"type": "Verse", "lyrics": "เดินเข้ามาใน hood มองไปเจอแต่ hater กูเริ่มจากศูนย์ now we gettin greater"},
    {"type": "Hook", "lyrics": "Flexin my money my crew gettin rich ไม่มีเวลามาฟังพวก opps ร้องไห้"}
]

print("🎧 --- What about My Lyrics: AI Analysis --- 🎧\n")

print("📊 Arrangement Insights:")
structure_feedback = evaluate_song_structure(mock_song_input)
for feedback in structure_feedback:
    print(feedback)

full_lyrics = " ".join([sec['lyrics'] for sec in mock_song_input])
vibe, detail_scores = analyze_vibe_score(full_lyrics)
print(f"\n🎵 Detected Vibe: {vibe}")
print(f"   Score Details: {detail_scores}")

print("\n🛠️ Producer Briefing:")
briefing = generate_producer_brief("hip-hop", "trap", vibe, None)
print(f"   - Recommended Tempo: {briefing['Target_Tempo']} BPM")
print(f"   - Target Key & Scale: Key {briefing['Target_Key']} {briefing['Target_Mode']}")

🎧 --- What about My Lyrics: AI Analysis --- 🎧

📊 Arrangement Insights:

🎵 Detected Vibe: Aggressive_Flex
   Score Details: {'Melancholy': 1, 'Upbeat_Party': 0, 'Aggressive_Flex': 6, 'Seductive_Romance': 0}

🛠️ Producer Briefing:
   - Recommended Tempo: 110 BPM
   - Target Key & Scale: Key 1 Minor
